In [1]:
import pandas as pd

df = pd.read_csv("hour.csv")

df.head()

   instant      dteday  season  yr  ...  windspeed  casual  registered  cnt
0        1  2011-01-01       1   0  ...        0.0       3          13   16
1        2  2011-01-01       1   0  ...        0.0       8          32   40
2        3  2011-01-01       1   0  ...        0.0       5          27   32
3        4  2011-01-01       1   0  ...        0.0       3          10   13
4        5  2011-01-01       1   0  ...        0.0       0           1    1

[5 rows x 17 columns]


In [2]:
df.shape

(17379, 17)


In [3]:
df.isna().sum()


instant       0
dteday        0
season        0
yr            0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64


In [4]:
df['cnt'].mean()

np.float64(189.46308763450142)


In [5]:
df['cnt'].describe()

count    17379.000000
mean       189.463088
std        181.387599
min          1.000000
25%         40.000000
50%        142.000000
75%        281.000000
max        977.000000
Name: cnt, dtype: float64


In [6]:
df.groupby('hr')['cnt'].mean()

hr
0      53.898072
1      33.375691
2      22.869930
3      11.727403
4       6.352941
5      19.889819
6      76.044138
7     212.064649
8     359.011004
9     219.309491
10    173.668501
11    208.143054
12    253.315934
13    253.661180
14    240.949246
15    251.233196
16    311.983562
17    461.452055
18    425.510989
19    311.523352
20    226.030220
21    172.314560
22    131.335165
23     87.831044
Name: cnt, dtype: float64


In [7]:
df['cnt'].plot()

<Axes: >


In [8]:
df.groupby('hr')['cnt'].mean().plot(kind='line')

<Axes: xlabel='hr'>


In [9]:
df.groupby('temp')['cnt'].mean().plot(kind='line')

<Axes: xlabel='temp'>


In [10]:
df.plot(kind='scatter',x='temp',y='cnt')



<Axes: xlabel='temp', ylabel='cnt'>


In [11]:
df.groupby('weekday')['cnt'].mean().plot(kind='bar')

<Axes: xlabel='weekday', ylabel='cnt'>


In [12]:
df.groupby('season')['cnt'].mean().plot(kind='bar')


<Axes: xlabel='season', ylabel='cnt'>


In [13]:
df.groupby('weathersit')['cnt'].mean().plot(kind='bar')

<Axes: xlabel='weathersit', ylabel='cnt'>


In [14]:
X=df[['hr','weekday','season']]
y = df['cnt']

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [16]:
# Install dependencies using requirements.txt before running this notebook.


In [17]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(13903, 3)
(3476, 3)
(13903,)
(3476,)


In [18]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()



In [19]:
model.fit(X_train, y_train)

LinearRegression()


In [20]:
y_pred = model.predict(X_test)

In [21]:
y_pred[:10]

array([192.33339749, 202.02980355, 211.72620962, 221.42261569,
       231.11902175, 240.81542782, 250.51183388, 260.20823995,
       269.90464602, 279.60105208])


In [22]:
from sklearn.metrics import mean_absolute_error
mae=mean_absolute_error(y_test,y_pred)

In [23]:
mae

150.71802956236994


In [24]:
baseline = y_train.mean()



In [25]:
baseline

np.float64(174.63914263108683)


In [26]:
import numpy as np

baseline_pred = np.full(len(y_test), baseline)
mae1 = mean_absolute_error(y_test,baseline_pred)

In [27]:
mae1

174.98460136882977


In [28]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)



In [29]:
rmse

np.float64(206.87967639482048)


In [30]:
mse_baseline = mean_squared_error(y_test, baseline_pred)
rmse_baseline = np.sqrt(mse_baseline)


In [31]:
rmse_baseline

np.float64(232.60837481037086)


In [32]:
# Install dependencies using requirements.txt before running this notebook.


In [33]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=0,
    allow_writing_files=False,
    verbose=False
)

cat_model.fit(X_train, y_train)

cat_pred = cat_model.predict(X_test)


In [34]:
mae_cat = mean_absolute_error(y_test,cat_pred)
mse_cat = mean_squared_error(y_test,cat_pred)
rmse_cat = np.sqrt(mse_cat)

In [35]:
mae_cat

96.52631320371347


In [36]:
rmse_cat

np.float64(139.5959105528474)


In [37]:
results = pd.DataFrame({
    'hr': X_test['hr'].to_numpy(),
    'actual': y_test.to_numpy(),
    'predicted': cat_pred
})


In [38]:
results['error'] = abs(results['actual'] - results['predicted'])


In [39]:
results.head()

   hr  actual   predicted       error
0  12     283  205.856501   77.143499
1  13     253  209.384472   43.615528
2  14     261  199.563400   61.436600
3  15     306  214.344656   91.655344
4  16     445  320.068030  124.931970


In [40]:
results.groupby('hr')['error'].mean().plot(kind='bar')


<Axes: xlabel='hr', ylabel='cnt'>


In [41]:
df['cnt_lag1'] = df['cnt'].shift(1)

df[['cnt', 'cnt_lag1']].head()

   cnt  cnt_lag1
0   16       NaN
1   40      16.0
2   32      40.0
3   13      32.0
4    1      13.0


In [42]:
df['datetime'] = pd.to_datetime(df['dteday']) + pd.to_timedelta(df['hr'], unit='h')

df[['dteday', 'hr', 'datetime']].head()

       dteday  hr            datetime
0  2011-01-01   0 2011-01-01 00:00:00
1  2011-01-01   1 2011-01-01 01:00:00
2  2011-01-01   2 2011-01-01 02:00:00
3  2011-01-01   3 2011-01-01 03:00:00
4  2011-01-01   4 2011-01-01 04:00:00


In [43]:
df['datetime'].diff().value_counts()

datetime
0 days 01:00:00    17303
0 days 02:00:00       64
0 days 03:00:00        6
0 days 13:00:00        1
0 days 23:00:00        1
0 days 07:00:00        1
0 days 14:00:00        1
1 days 13:00:00        1
Name: count, dtype: int64


In [44]:
gap = df['datetime'].diff() != pd.Timedelta(hours=1)

df.loc[gap, 'cnt_lag1'] = np.nan

In [45]:
df['cnt_lag1'].isna().sum()

np.int64(76)


In [46]:
df_model = df.dropna(subset=['cnt_lag1']).copy()

X_new = df_model[['hr', 'weekday', 'season', 'cnt_lag1']]
y_new = df_model['cnt']

print(X_new.shape)
print(y_new.shape)

(17303, 4)
(17303,)


In [47]:
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y_new, test_size=0.2, shuffle=False
)

In [48]:
cat_model_new = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=0,
    allow_writing_files=False,
    verbose=False
)

cat_model_new.fit(X_train_new, y_train_new)

cat_pred_new = cat_model_new.predict(X_test_new)

In [49]:
mae_new = mean_absolute_error(y_test_new, cat_pred_new)
rmse_new = np.sqrt(mean_squared_error(y_test_new, cat_pred_new))
print("CatBoost with lag MAE:", mae_new)
print("CatBoost with lag RMSE:", rmse_new)


CatBoost with lag MAE: 32.12994097636786
CatBoost with lag RMSE: 51.62572564625998


In [50]:
results_new = pd.DataFrame({
    'hr': X_test_new['hr'].to_numpy(),
    'actual':y_test_new.to_numpy(),
    'predicted': cat_pred_new
})


In [51]:
results_new['error'] = abs(results_new['actual'] - results_new['predicted'])

In [52]:
results_new.groupby('hr')['error'].mean().plot(kind='bar')

<Axes: xlabel='hr', ylabel='cnt'>


In [53]:
results_new.groupby('hr')['error'].mean().sort_values(ascending=False)
results_new.groupby('hr')['error'].mean().sort_values(ascending=False).head(5)

hr
17    91.538008
8     86.584717
16    56.615867
18    47.550324
12    42.704933
Name: error, dtype: float64


In [54]:

X_train_old = X_train_new.drop(columns=['cnt_lag1'])
X_test_old = X_test_new.drop(columns=['cnt_lag1'])

# Обучаем CatBoost без cnt_lag1
cat_model_old = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=0,
    allow_writing_files=False,
    verbose=False
)

cat_model_old.fit(X_train_old, y_train_new)

# Получаем предсказания
cat_pred_old = cat_model_old.predict(X_test_old)

# Сравниваем ошибки
mae_old = mean_absolute_error(y_test_new, cat_pred_old)
rmse_old = np.sqrt(mean_squared_error(y_test_new, cat_pred_old))

print("Без cnt_lag1:")
print("MAE:", mae_old)
print("RMSE:", rmse_old)

print("\nС cnt_lag1:")
print("MAE:", mae_new)
print("RMSE:", rmse_new)

Без cnt_lag1:
MAE: 96.39327876325845
RMSE: 139.57362876002722

С cnt_lag1:
MAE: 32.12994097636786
RMSE: 51.62572564625998


In [55]:
def predict_demand(hr,weekday,season,cnt_lag1):
    data = pd.DataFrame({
        'hr': [hr],
        'weekday': [weekday],
        'season': [season],
        'cnt_lag1': [cnt_lag1]
    })
    prediction = cat_model_new.predict(data)
    return prediction[0]




In [56]:
predict_demand(17, 2, 3, 400)

np.float64(733.2613316583656)


In [57]:
cat_model_new.save_model("bike_demand_catboost.cbm")

In [58]:
loaded_model = CatBoostRegressor()
loaded_model.load_model("bike_demand_catboost.cbm")




CatBoostRegressor(allow_writing_files=False, depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=0, verbose=0)


In [59]:
loaded_model.predict(pd.DataFrame({
    'hr': [17],
    'weekday': [2],
    'season': [3],
    'cnt_lag1': [400]
}))

array([733.26133166])
